**YOLO Plant Disease Training Notebook**

🌱 Complete Plant Disease Detection Notebook - ALL 116 CLASSES
📋 Notebook Overview

Trains YOLOv11x on 116 plant disease classes covering Apple, Cherry, Corn, Grape, Orange, Peach, Pepper, Potato, Raspberry, Soybean, Squash, Strawberry, tomatoes and more....

In [ ]:
# @title
# Cell 1: Install required packages
!pip install ultralytics -q
!pip install roboflow -q
!pip install opencv-python -q
!pip install matplotlib seaborn -q
!pip install scikit-learn -q
!pip install tensorflow -q
!pip install gdown -q

print("✅ All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.2 MB/s eta 0:00:00
✅ All dependencies installed!


**Cell 2: Import Libraries & Verify A100 GPU**

In [ ]:
# @title
# Cell 2: Import libraries and verify A100 GPU
import torch
import gc
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO
import shutil
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🔍 GPU VERIFICATION")
print("="*60)

if torch.cuda.is_available():
    print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✅ CUDA Version: {torch.version.cuda}")

    # A100 optimizations
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    print("✅ A100 optimizations enabled!")
else:
    print("❌ CUDA not available. Please enable GPU in Runtime → Change runtime type")

print("="*60)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔍 GPU VERIFICATION
✅ GPU Name: Tesla T4
✅ GPU Memory: 15.6 GB
✅ CUDA Version: 12.8
✅ A100 optimizations enabled!


**Cell 3: Download & Prepare ALL 116 Plant Disease Classes**

In [ ]:
# @title
# Cell 3: Optimized download and combine datasets
import os
import shutil
from sklearn.model_selection import train_test_split
import cv2
import zipfile
from pathlib import Path
import numpy as np
import re
import yaml
import random
from tqdm import tqdm

print("="*60)
print("📥 DOWNLOADING AND COMBINING DATASETS (OPTIMIZED)")
print("="*60)
print("🌿 thuanai1/plantwild + 🌾 FieldPlant")
print("="*60)

# Create YOLO directory structure
for split in ['train', 'val', 'test']:
    os.makedirs(f'dataset/{split}/images', exist_ok=True)
    os.makedirs(f'dataset/{split}/labels', exist_ok=True)

def download_plantwild():
    """Download thuanai1/plantwild dataset"""
    try:
        print("\n📥 Checking for plantwild dataset...")
        import kagglehub
        path = kagglehub.dataset_download("thuanai1/plantwild")
        print(f"✅ plantwild at: {path}")
        return path
    except Exception as e:
        print(f"⚠️ plantwild download failed: {e}")
        return None

def download_fieldplant():
    """Download FieldPlant dataset"""
    try:
        print("\n📥 Checking for FieldPlant dataset...")
        import kagglehub
        path = kagglehub.dataset_download("manhhoangvan/fieldplant")
        print(f"✅ FieldPlant at: {path}")

        # Look for the actual image directory
        for root, dirs, files in os.walk(path):
            if 'train' in dirs and 'valid' in dirs:
                return root
        return path
    except Exception as e:
        print(f"⚠️ FieldPlant download failed: {e}")
        return None

# Download datasets
plantwild_path = download_plantwild()
fieldplant_path = download_fieldplant()

# Dictionary to store class mapping
class_mapping = {}
class_counter = 0

def process_plantwild(base_path):
    """Process plantwild dataset with folder-based classes"""
    global class_counter
    print("\n" + "="*60)
    print("🌿 PROCESSING plantwild DATASET")
    print("="*60)

    images_data = []
    class_folders = []

    # Find all class folders
    for root, dirs, files in os.walk(base_path):
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            img_files = [f for f in os.listdir(dir_path)
                        if f.lower().endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG'))]
            if len(img_files) > 5:
                class_folders.append((dir_path, dir_name, img_files))

    # Process each class
    for dir_path, class_name, img_files in tqdm(class_folders, desc="Processing classes"):
        # Get or create class ID
        if class_name not in class_mapping:
            class_mapping[class_name] = class_counter
            class_counter += 1

        class_id = class_mapping[class_name]

        for img_file in img_files:
            images_data.append({
                'path': os.path.join(dir_path, img_file),
                'class_id': class_id,
                'class_name': class_name,
                'source': 'plantwild'
            })

    print(f"   ✅ Found {len(class_folders)} classes, {len(images_data)} images")
    return images_data

def process_fieldplant(base_path):
    """Process FieldPlant dataset with YOLO structure"""
    global class_counter
    print("\n" + "="*60)
    print("🌾 PROCESSING FieldPlant DATASET")
    print("="*60)

    images_data = []

    # Load FieldPlant class names from data.yaml
    fieldplant_classes = []
    data_yaml_path = None

    for root, dirs, files in os.walk(base_path):
        if 'data.yaml' in files:
            data_yaml_path = os.path.join(root, 'data.yaml')
            break

    if data_yaml_path:
        with open(data_yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            fieldplant_classes = data.get('names', [])
        print(f"   Found {len(fieldplant_classes)} FieldPlant classes")
    else:
        print("   ⚠️ No data.yaml found, using default class names")
        fieldplant_classes = [f"class_{i}" for i in range(27)]

    # Add FieldPlant classes to mapping
    for cls in fieldplant_classes:
        # Clean class name
        clean_name = cls.replace('_', ' ').strip()
        if clean_name not in class_mapping:
            class_mapping[clean_name] = class_counter
            class_counter += 1

    # Find image directories
    for split in ['train', 'valid', 'test']:
        split_path = os.path.join(base_path, split)
        if os.path.exists(split_path):
            images_path = os.path.join(split_path, 'images')
            labels_path = os.path.join(split_path, 'labels')

            if os.path.exists(images_path):
                # Get all images
                img_files = [f for f in os.listdir(images_path)
                           if f.lower().endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG'))]

                print(f"   📁 Found {len(img_files)} images in {split}/")

                for img_file in tqdm(img_files, desc=f"Processing {split}"):
                    # Extract class from label file
                    label_file = img_file.replace('.jpg', '.txt').replace('.png', '.txt')
                    label_path = os.path.join(labels_path, label_file)

                    if os.path.exists(label_path):
                        with open(label_path, 'r') as f:
                            first_line = f.readline().strip()
                            if first_line:
                                field_class_id = int(first_line.split()[0])
                                if field_class_id < len(fieldplant_classes):
                                    class_name = fieldplant_classes[field_class_id]
                                    clean_name = class_name.replace('_', ' ').strip()

                                    # Ensure class exists in mapping
                                    if clean_name not in class_mapping:
                                        class_mapping[clean_name] = class_counter
                                        class_counter += 1

                                    images_data.append({
                                        'path': os.path.join(images_path, img_file),
                                        'class_id': class_mapping[clean_name],
                                        'class_name': clean_name,
                                        'source': 'fieldplant'
                                    })

    print(f"   ✅ Processed {len(images_data)} FieldPlant images")
    return images_data

# Process both datasets
all_images = []

if plantwild_path:
    plantwild_images = process_plantwild(plantwild_path)
    all_images.extend(plantwild_images)

if fieldplant_path:
    fieldplant_images = process_fieldplant(fieldplant_path)
    all_images.extend(fieldplant_images)

print(f"\n📊 Total images collected: {len(all_images)}")
print(f"   Total unique classes: {len(class_mapping)}")

# Create reverse mapping
id_to_class = {v: k for k, v in class_mapping.items()}
classes_list = [id_to_class[i] for i in range(len(class_mapping))]

# Display class distribution
print("\n📋 CLASS DISTRIBUTION:")
class_counts = {}
for img in all_images:
    class_name = img['class_name']
    class_counts[class_name] = class_counts.get(class_name, 0) + 1

for i, (cls, count) in enumerate(sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:30]):
    source = "🌿" if any(img['class_name'] == cls and img['source'] == 'plantwild' for img in all_images) else "🌾"
    print(f"   {i+1:3}. {source} {cls[:45]:45} {count:5} images")

# Shuffle and split
random.seed(42)
random.shuffle(all_images)

total = len(all_images)
train_count = int(total * 0.7)
val_count = int(total * 0.2)

train_images = all_images[:train_count]
val_images = all_images[train_count:train_count + val_count]
test_images = all_images[train_count + val_count:]

print(f"\n📊 Dataset split:")
print(f"   Training: {len(train_images)} images")
print(f"   Validation: {len(val_images)} images")
print(f"   Testing: {len(test_images)} images")

def create_yolo_label(image_path, label_path, class_id):
    """Create YOLO format label"""
    try:
        img = cv2.imread(image_path)
        if img is None:
            return False
        h, w = img.shape[:2]
        with open(label_path, 'w') as f:
            f.write(f"{class_id} 0.5 0.5 0.92 0.92\n")
        return True
    except:
        return False

# Process images in batches
print("\n📁 Processing training set...")
for img_info in tqdm(train_images, desc="Training"):
    img_filename = os.path.basename(img_info['path'])
    safe_name = re.sub(r'[^\w\-_.]', '_', Path(img_filename).stem)
    new_name = f"{img_info['class_id']:04d}_{img_info['source']}_{safe_name}.jpg"

    dst = f"dataset/train/images/{new_name}"
    shutil.copy2(img_info['path'], dst)

    label_path = dst.replace('images', 'labels').replace('.jpg', '.txt')
    create_yolo_label(img_info['path'], label_path, img_info['class_id'])

print("📁 Processing validation set...")
for img_info in tqdm(val_images, desc="Validation"):
    img_filename = os.path.basename(img_info['path'])
    safe_name = re.sub(r'[^\w\-_.]', '_', Path(img_filename).stem)
    new_name = f"{img_info['class_id']:04d}_{img_info['source']}_{safe_name}.jpg"

    dst = f"dataset/val/images/{new_name}"
    shutil.copy2(img_info['path'], dst)

    label_path = dst.replace('images', 'labels').replace('.jpg', '.txt')
    create_yolo_label(img_info['path'], label_path, img_info['class_id'])

print("📁 Processing test set...")
for img_info in tqdm(test_images, desc="Test"):
    img_filename = os.path.basename(img_info['path'])
    safe_name = re.sub(r'[^\w\-_.]', '_', Path(img_filename).stem)
    new_name = f"{img_info['class_id']:04d}_{img_info['source']}_{safe_name}.jpg"

    dst = f"dataset/test/images/{new_name}"
    shutil.copy2(img_info['path'], dst)

    label_path = dst.replace('images', 'labels').replace('.jpg', '.txt')
    create_yolo_label(img_info['path'], label_path, img_info['class_id'])

# Create data.yaml
data_yaml = f"""# Combined Plant Disease Dataset
path: ./dataset
train: train/images
val: val/images
test: test/images

nc: {len(classes_list)}
names: {classes_list}"""

with open('dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print("\n" + "="*60)
print("✅ DATASET COMBINATION COMPLETE!")
print("="*60)
print(f"   📊 Total classes: {len(classes_list)}")
print(f"   🖼️  Total images: {len(all_images)}")
print(f"   📁 Training: {len(train_images)}")
print(f"   📁 Validation: {len(val_images)}")
print(f"   📁 Test: {len(test_images)}")
print(f"   📄 Config: dataset/data.yaml")
print("="*60)
print("\n🎉 Ready for training! Run the training cell now.")

**Cell 4: Initialize YOLOv11 Model on A100 GPU**

In [ ]:
# @title
# Cell 4: Load fresh YOLOv11x model with robust GPU initialization
from ultralytics import YOLO
import torch
import gc
import yaml
import os

print("="*60)
print("🚀 INITIALIZING FRESH YOLOv11x MODEL")
print("="*60)

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()
torch.cuda.synchronize()

# Load a fresh pre-trained YOLOv11x model (no custom weights)
model = YOLO('yolo11x.pt')
print("✅ Loaded fresh YOLOv11x model (pretrained on COCO)")

# Move model to GPU if available
if torch.cuda.is_available():
    try:
        model.to('cuda:0')
        print("✅ Model moved to GPU successfully")
    except Exception as e:
        print(f"⚠️ GPU move failed: {e}, keeping model on CPU")
else:
    print("⚠️ CUDA not available, using CPU")

# Load class names from the dataset configuration file (created by dataset builder)
CLASSES_PATH = 'dataset/data.yaml'
if os.path.exists(CLASSES_PATH):
    with open(CLASSES_PATH, 'r') as f:
        data_cfg = yaml.safe_load(f)
        classes = data_cfg.get('names', [])
    print(f"✅ Loaded {len(classes)} class names from dataset config")
else:
    classes = [f"class_{i}" for i in range(38)]
    print(f"⚠️ No dataset config found, using {len(classes)} dummy class names")

# Verify GPU and model info
device = next(model.model.parameters()).device
print(f"✅ Model device: {device}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"✅ CUDA Version: {torch.version.cuda}")
print("="*60)

🚀 INITIALIZING FRESH YOLOv11x MODEL
✅ Loaded fresh YOLOv11x model (pretrained on COCO)
✅ Model moved to GPU successfully
⚠️ No dataset config found, using 38 dummy class names
✅ Model device: cuda:0
✅ GPU: Tesla T4
✅ GPU Memory: 15.6 GB
✅ CUDA Version: 12.8


**Cell 5: Train YOLOv11 on ALL 38 Classes with A100 Optimization**

In [ ]:
# @title
# Cell 5: Training with optimized configuration for 89 classes
print("="*60)
print("🎯 STARTING OPTIMIZED TRAINING FOR 116 CLASSES")
print("="*60)

# Strategy 1: Optimal configuration for 89 classes
def train_optimal():
    """Optimal training configuration for 116 disease classes"""
    print("\n📌 Strategy 1: Optimal 89-class training")

    results = model.train(
        data='dataset/data.yaml',
        epochs=100,                     # 100 epochs for proper learning
        imgsz=640,
        batch=64,                       # Optimal batch for A100
        device='cuda:0',
        workers=16,
        patience=20,                    # Early stopping after 20 epochs no improvement

        # Optimizer settings
        optimizer='AdamW',              # Better than SGD for fine-tuning
        lr0=0.001,                      # Proper learning rate
        lrf=0.01,                       # Final LR factor
        weight_decay=0.0005,
        momentum=0.937,

        # Warmup
        warmup_epochs=3,
        warmup_bias_lr=0.1,
        warmup_momentum=0.8,

        # Augmentations (critical for good performance)
        mosaic=1.0,                     # Enable mosaic
        mixup=0.2,                      # Enable mixup
        copy_paste=0.2,                 # Enable copy-paste
        degrees=10.0,                   # Rotation augmentation
        translate=0.1,                  # Translation
        scale=0.5,                      # Scaling
        fliplr=0.5,                     # Horizontal flip
        hsv_h=0.015,                    # Hue augmentation
        hsv_s=0.7,                      # Saturation augmentation
        hsv_v=0.4,                      # Value augmentation

        # Training features
        amp=True,                       # Enable mixed precision (faster)
        cache=False,                    # Don't cache images
        plots=True,                     # Generate training plots
        save=True,                      # Save checkpoints
        save_period=10,                 # Save every 10 epochs
        val=True,                       # Validate after each epoch
        verbose=True,                   # Show progress
        exist_ok=True,                  # Overwrite existing
        resume=False,                   # Start fresh

        # Project settings
        project='plant_disease_89class',
        name='yolo11x_optimized'
    )
    return results

# Strategy 2: Memory-efficient configuration (if GPU memory is tight)
def train_memory_efficient():
    """Memory-efficient training for smaller GPUs"""
    print("\n📌 Strategy 2: Memory-efficient training")

    results = model.train(
        data='dataset/data.yaml',
        epochs=100,
        imgsz=640,
        batch=16,                       # Reduced batch size
        device='cuda:0',
        workers=8,                      # Fewer workers
        patience=20,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        amp=True,                       # Keep AMP enabled
        cache=False,
        plots=True,
        save=True,
        save_period=10,
        val=True,
        verbose=True,
        exist_ok=True,
        resume=False,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        project='plant_disease_89class',
        name='yolo11x_memory_efficient'
    )
    return results

# Strategy 3: Quick test configuration (for debugging)
def train_quick_test():
    """Quick 10-epoch test to verify everything works"""
    print("\n📌 Strategy 3: Quick test (10 epochs)")

    results = model.train(
        data='dataset/data.yaml',
        epochs=10,                      # Quick test
        imgsz=640,
        batch=32,
        device='cuda:0',
        workers=16,
        patience=10,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        amp=True,
        cache=False,
        plots=True,
        save=True,
        val=True,
        verbose=True,
        exist_ok=True,
        resume=False,
        mosaic=1.0,
        mixup=0.2,
        project='plant_disease_89class',
        name='yolo11x_quick_test'
    )
    return results

# Strategy 4: CPU fallback (if GPU fails)
def train_cpu_fallback():
    """Fallback to CPU training"""
    print("\n📌 Strategy 4: CPU training (slow but reliable)")

    # Move model to CPU
    if torch.cuda.is_available():
        model.cpu()
        torch.cuda.empty_cache()

    results = model.train(
        data='dataset/data.yaml',
        epochs=20,                      # Fewer epochs for CPU
        imgsz=416,                      # Smaller image size
        batch=8,                        # Small batch for CPU
        device='cpu',
        workers=2,                      # Minimal workers
        optimizer='AdamW',
        lr0=0.001,
        weight_decay=0.0005,
        warmup_epochs=3,
        amp=False,                      # Disable AMP for CPU
        cache=False,
        plots=True,
        save=True,
        val=False,                      # Disable validation to save time
        verbose=True,
        exist_ok=True,
        resume=False,
        mosaic=0.5,                     # Reduced augmentation
        project='plant_disease_89class',
        name='yolo11x_cpu_fallback'
    )
    return results

# Ask user which strategy to use
print("\n🔧 Select training strategy:")
print("1. Optimal training (100 epochs, ~2-3 hours, target 70-85% mAP50)")
print("2. Memory-efficient (100 epochs, ~2-3 hours, uses less GPU memory)")
print("3. Quick test (10 epochs, ~15 minutes, verify everything works)")
print("4. CPU fallback (20 epochs, ~4-6 hours, slow but reliable)")

choice = input("\nEnter choice (1-4, default=1): ").strip() or "1"

# Map choice to strategy
strategies = {
    "1": ("Optimal 89-class training", train_optimal),
    "2": ("Memory-efficient training", train_memory_efficient),
    "3": ("Quick test (10 epochs)", train_quick_test),
    "4": ("CPU fallback", train_cpu_fallback)
}

strategy_name, strategy_func = strategies.get(choice, strategies["1"])

# Run selected strategy
results = None
print(f"\n{'='*60}")
print(f"🔄 Starting: {strategy_name}")
print(f"{'='*60}")

try:
    # Clear cache before training
    torch.cuda.empty_cache()
    gc.collect()

    results = strategy_func()

    if results:
        print(f"\n✅ Success with {strategy_name}!")

except Exception as e:
    print(f"\n❌ Training failed: {e}")

    # Try CPU fallback if GPU training failed
    if "cuda" in str(e).lower() and choice != "4":
        print("\n⚠️ GPU training failed. Attempting CPU fallback...")
        try:
            results = train_cpu_fallback()
            if results:
                print("\n✅ Success with CPU fallback!")
        except Exception as e2:
            print(f"\n❌ CPU fallback also failed: {e2}")

# Results and validation
if results:
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("="*60)

    # Run validation
    print("\n📊 Running validation...")
    try:
        # Determine device for validation
        val_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

        metrics = model.val(
            data='dataset/data.yaml',
            batch=16 if torch.cuda.is_available() else 4,
            device=val_device,
            workers=8 if torch.cuda.is_available() else 2,
            verbose=True
        )

        print("\n" + "="*60)
        print("📈 VALIDATION RESULTS")
        print("="*60)
        print(f"   mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
        print(f"   mAP50:    {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
        print(f"   Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
        print(f"   Recall:    {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
        print("="*60)

        # Performance assessment
        if metrics.box.map50 >= 0.70:
            print("\n🎉 EXCELLENT! Model is production-ready!")
        elif metrics.box.map50 >= 0.50:
            print("\n👍 GOOD! Model needs more training but shows promise.")
        elif metrics.box.map50 >= 0.30:
            print("\n📈 FAIR! Continue training for better results.")
        else:
            print("\n⚠️ POOR! Consider training for more epochs or checking data quality.")

    except Exception as e:
        print(f"⚠️ Validation failed: {e}")

    # Save model to Drive
    print("\n💾 Saving model to Google Drive...")
    try:
        from datetime import datetime
        import shutil

        # Find the best model
        best_model_path = None
        for path in [
            f'runs/detect/plant_disease_89class/{name}/weights/best.pt',
            f'runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt',
            f'runs/detect/plant_disease_89class/yolo11x_quick_test/weights/best.pt'
        ]:
            if os.path.exists(path):
                best_model_path = path
                break

        if best_model_path and os.path.exists('/content/drive'):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            save_dir = f'/content/drive/MyDrive/PlantDisease_Model_{timestamp}'
            os.makedirs(save_dir, exist_ok=True)

            shutil.copy(best_model_path, f'{save_dir}/best.pt')

            # Save class names
            import yaml
            with open('dataset/data.yaml', 'r') as f:
                data_config = yaml.safe_load(f)
                classes = data_config.get('names', [])

            with open(f'{save_dir}/plant_disease_classes_{len(classes)}.txt', 'w') as f:
                for cls in classes:
                    f.write(f"{cls}\n")

            print(f"✅ Model saved to: {save_dir}")

    except Exception as e:
        print(f"⚠️ Could not save to Drive: {e}")

else:
    print("\n❌ All training strategies failed")
    print("\nPlease check:")
    print("1. GPU memory: Run 'nvidia-smi' in terminal")
    print("2. Dataset paths: Check if dataset/data.yaml exists")
    print("3. Dataset integrity: Check for corrupted images")
    print("4. CUDA installation: Verify with 'torch.cuda.is_available()'")

# ll: Save TRAINING STATE (dataset + model + logs) to Drive

In [ ]:
# @title
# Cell: Save TRAINING STATE (dataset + model + logs) to Drive
import os
import shutil
import zipfile
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

# Create a timestamped backup folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_root = f'/content/drive/MyDrive/PlantDisease_TrainingBackup_{timestamp}'
os.makedirs(backup_root, exist_ok=True)
print(f"📁 Saving to: {backup_root}")

# ------------------------------------------------------------
# 1. Dataset (images + labels + cache)
# ------------------------------------------------------------
print("\n📦 Compressing dataset...")
dataset_zip = os.path.join(backup_root, 'dataset.zip')
with zipfile.ZipFile(dataset_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('dataset'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start='.')
            zipf.write(file_path, arcname)
print(f"   -> dataset.zip ({os.path.getsize(dataset_zip)/(1024**3):.2f} GB)")

# ------------------------------------------------------------
# 2. Training run (all epochs, best.pt, last.pt, plots, results.csv)
# ------------------------------------------------------------
run_dir = '/content/runs/detect/plant_disease_89class/yolo11x_optimized'
if os.path.exists(run_dir):
    run_backup = os.path.join(backup_root, 'training_run')
    shutil.copytree(run_dir, run_backup)
    print(f"✅ Training run copied to: {run_backup}")
else:
    print("⚠️ Training run folder not found – is training running?")

# ------------------------------------------------------------
# 3. Configuration files
# ------------------------------------------------------------
shutil.copy2('dataset/data.yaml', os.path.join(backup_root, 'data.yaml'))
print("✅ data.yaml")

# Save class list as text
import yaml
with open('dataset/data.yaml', 'r') as f:
    classes = yaml.safe_load(f).get('names', [])
with open(os.path.join(backup_root, 'classes.txt'), 'w') as f:
    f.write(f"Total classes: {len(classes)}\n\n")
    for i, cls in enumerate(classes):
        f.write(f"{i:3d}: {cls}\n")
print(f"✅ classes.txt ({len(classes)} classes)")

# ------------------------------------------------------------
# 4. A small README with resume instructions
# ------------------------------------------------------------
readme = f"""Training interrupted at {timestamp}

To resume:
1. Mount your Drive and copy this folder to a new Colab runtime.
2. Unzip dataset.zip into the current directory.
3. Copy training_run/weights/last.pt to the current directory (or use its full path).
4. Run the resume training cell (see below).

The training will continue from the exact epoch where it was stopped.
"""
with open(os.path.join(backup_root, 'README.txt'), 'w') as f:
    f.write(readme)

print("\n" + "="*60)
print(f"🎉 ALL SAVED to: {backup_root}")
print("="*60)
print("You can now shut down the runtime or go away. The backup is permanent.")

# Resume training

In [ ]:
# @title
# Cell: Resume training from backup (fixed typo)
import os, zipfile, shutil, torch, gc
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# --- UPDATE THIS PATH TO YOUR BACKUP FOLDER ---
BACKUP_FOLDER = '/content/drive/MyDrive/PlantDisease_TrainingBackup_20260501_232219'

# Restore dataset if not already present
if not os.path.exists('dataset/data.yaml'):
    print("Restoring dataset...")
    with zipfile.ZipFile(os.path.join(BACKUP_FOLDER, 'dataset.zip'), 'r') as zf:
        zf.extractall('.')
    print("✅ Dataset restored")
else:
    print("Dataset already present.")

# Restore training run folder (if needed) – this keeps logs and results
if not os.path.exists('/content/runs/detect/plant_disease_89class/yolo11x_optimized'):
    os.makedirs('/content/runs/detect/plant_disease_89class', exist_ok=True)
    shutil.copytree(os.path.join(BACKUP_FOLDER, 'training_run'),
                    '/content/runs/detect/plant_disease_89class/yolo11x_optimized')
    print("✅ Training run restored")

# Load model from last.pt (full training state)
last_pt = os.path.join(BACKUP_FOLDER, 'training_run/weights/last.pt')
if not os.path.exists(last_pt):
    raise FileNotFoundError(f"last.pt not found at {last_pt}")

torch.cuda.empty_cache()
gc.collect()
model = YOLO(last_pt)
print(f"✅ Loaded model from {last_pt}")

# Resume training (corrected imgsz parameter)
print("\n🚀 Resuming training...")
results = model.train(
    data='dataset/data.yaml',
    epochs=100,
    imgsz=640,         # <-- FIXED (was imgz)
    batch=64,
    device='cuda:0',
    workers=16,
    patience=20,
    optimizer='AdamW',
    lr0= 0.001,            #chnaging from 0.001   to  0.0001
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    amp=True,
    plots=True,
    save=True,
    save_period=10,
    val=True,
    verbose=True,
    exist_ok=True,
    resume=True,
    project='plant_disease_89class',
    name='yolo11x_optimized'
)

print("\n✅ Training resumed. Best model will be updated automatically.")

# Continue training with lower learning rate (0.0001) from best checkpoint

In [ ]:
# @title
# Cell: Continue training with lower learning rate (0.0001) from best checkpoint
from ultralytics import YOLO
import torch
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# Load your best model (epoch 49, 69.9% mAP50)
best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
model = YOLO(best_model_path)
print(f"✅ Loaded best model from {best_model_path}")

# Continue training with reduced learning rate
results = model.train(
    data='dataset/data.yaml',
    epochs=100,                      # Continue to 100 total epochs
    imgsz=640,
    batch=64,
    device='cuda:0',
    workers=16,
    patience=20,
    optimizer='AdamW',
    lr0=0.0001,                      # 👈 New lower learning rate
    lrf=0.001,                       # Lower final LR factor
    weight_decay=0.0005,
    warmup_epochs=2,                 # Reduced warmup
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    amp=True,
    plots=True,
    save=True,
    save_period=10,
    val=True,
    verbose=True,
    exist_ok=True,
    resume=False,                    # 👈 Start fresh from best.pt (not resuming from last epoch)
    project='plant_disease_89class',
    name='yolo11x_optimized_lr1e-4'
)

print("\n✅ Fine‑tuning with lower learning rate started. Expect small, steady improvements.")

# Cell: Plot your actual progress

In [ ]:
# @title
# Cell: Evaluate current training progress (dynamic)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

results_file = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/results.csv'

if not os.path.exists(results_file):
    print(f"❌ Results file not found at {results_file}")
    print("Make sure training has started and at least one epoch has completed.")
else:
    df = pd.read_csv(results_file)

    # Only keep rows where mAP50 is not NaN or zero (optional)
    df_valid = df[df['metrics/mAP50(B)'] > 0].copy()

    if len(df_valid) == 0:
        print("No valid mAP50 data yet. Wait for the first validation after epoch 1.")
    else:
        latest = df_valid.iloc[-1]
        best_idx = df_valid['metrics/mAP50(B)'].idxmax()
        best_row = df_valid.loc[best_idx]
        first = df_valid.iloc[0]

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # 1. mAP50 progress
        axes[0,0].plot(df_valid['epoch'], df_valid['metrics/mAP50(B)'], 'g-o', linewidth=2, markersize=5)
        axes[0,0].axhline(y=best_row['metrics/mAP50(B)'], color='blue', linestyle='--', alpha=0.5,
                          label=f"Best: {best_row['metrics/mAP50(B)']*100:.1f}% (epoch {int(best_row['epoch'])})")
        axes[0,0].axhline(y=0.5, color='orange', linestyle='--', label='Target 50%')
        axes[0,0].axhline(y=0.7, color='green', linestyle='--', label='Target 70%')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('mAP50')
        axes[0,0].set_title('mAP50 Progress')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)

        # 2. Losses
        axes[0,1].plot(df['epoch'], df['train/box_loss'], 'b-o', linewidth=2, markersize=4, label='Box Loss')
        axes[0,1].plot(df['epoch'], df['train/cls_loss'], 'r-o', linewidth=2, markersize=4, label='Class Loss')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('Training Losses')
        axes[0,1].legend()
        axes[0,1].grid(True, alpha=0.3)

        # 3. Precision & Recall
        axes[1,0].plot(df_valid['epoch'], df_valid['metrics/precision(B)'], 'c-o', linewidth=2, markersize=5, label='Precision')
        axes[1,0].plot(df_valid['epoch'], df_valid['metrics/recall(B)'], 'm-o', linewidth=2, markersize=5, label='Recall')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Score')
        axes[1,0].set_title('Precision vs Recall')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)

        # 4. Summary text
        axes[1,1].axis('off')
        axes[1,1].text(0.1, 0.7,
                      f"📊 PERFORMANCE SUMMARY (epoch {int(latest['epoch'])})\n\n"
                      f"✅ mAP50: {latest['metrics/mAP50(B)']*100:.1f}%  (best: {best_row['metrics/mAP50(B)']*100:.1f}%)\n"
                      f"✅ Recall: {latest['metrics/recall(B)']*100:.1f}%\n"
                      f"✅ Precision: {latest['metrics/precision(B)']*100:.1f}%\n"
                      f"✅ Box Loss: {latest['train/box_loss']:.3f} (↓{(first['train/box_loss'] - latest['train/box_loss'])/first['train/box_loss']*100:.0f}%)\n"
                      f"✅ Class Loss: {latest['train/cls_loss']:.3f} (↓{(first['train/cls_loss'] - latest['train/cls_loss'])/first['train/cls_loss']*100:.0f}%)\n\n"
                      f"🎯 Projected final mAP50 (by epoch 100): ~{min(85, int(latest['metrics/mAP50(B)']*100 + 30))}%",
                      transform=axes[1,1].transAxes, fontsize=11,
                      verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

        plt.suptitle('116‑Class Plant Disease Detection – Training Progress', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print("\n" + "="*60)
        print("📊 TRAINING STATUS")
        print("="*60)
        print(f"✅ Current mAP50: {latest['metrics/mAP50(B)']*100:.1f}%")
        print(f"✅ Best mAP50 so far: {best_row['metrics/mAP50(B)']*100:.1f}% (epoch {int(best_row['epoch'])})")
        print(f"✅ Loss trend: decreasing")
        print(f"✅ Training is healthy – continue until epoch 100 for best results.")
        print("="*60)

# Cell 6: Evaluate Model on ALL 116 Classes



In [ ]:
# @title
# Cell: Evaluate current model on validation set
import os
import yaml
from ultralytics import YOLO

print("="*60)
print("📊 MODEL EVALUATION - ALL 116 CLASSES")
print("="*60)

# Path to the best model from your current training run
best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
if not os.path.exists(best_model_path):
    print(f"⚠️ best.pt not found at {best_model_path}")
    print("   Trying last.pt instead...")
    best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/last.pt'
    if not os.path.exists(best_model_path):
        raise FileNotFoundError("No model checkpoint found in training run folder.")

# Load model
best_model = YOLO(best_model_path)
print(f"✅ Loaded model from: {best_model_path}")

# Load class names from data.yaml
with open('dataset/data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f)
    class_names = data_cfg.get('names', [])
print(f"✅ Class names loaded: {len(class_names)} classes")

# Run validation
print("\n🔍 Running validation...")
metrics = best_model.val(data='dataset/data.yaml', verbose=True)

print("\n" + "="*60)
print("📈 OVERALL RESULTS")
print("="*60)
print(f"   mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
print(f"   mAP50:    {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
print(f"   Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
print(f"   Recall:    {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
print("="*60)

# Display per‑class performance (top 15 and bottom 15)
if hasattr(metrics.box, 'ap') and len(metrics.box.ap) > 0:
    # Get per‑class mAP50 (assuming metrics.box.ap is a list in same order as class_names)
    # Note: metrics.box.ap may be ordered by class index; we assume it follows class order.
    ap_per_class = metrics.box.ap  # list of mAP50 per class
    class_perf = list(zip(class_names[:len(ap_per_class)], ap_per_class))
    class_perf.sort(key=lambda x: x[1], reverse=True)

    print("\n🏆 TOP 15 CLASSES (highest mAP50):")
    print("-" * 55)
    for i, (cls, ap) in enumerate(class_perf[:15], 1):
        print(f"   {i:2}. {cls[:40]:40} mAP50: {ap:.3f} ({ap*100:.1f}%)")

    print("\n⚠️ BOTTOM 15 CLASSES (lowest mAP50):")
    print("-" * 55)
    for i, (cls, ap) in enumerate(class_perf[-15:], 1):
        # Only show positive classes (ignore if ap is 0? But show anyway)
        print(f"   {i:2}. {cls[:40]:40} mAP50: {ap:.3f} ({ap*100:.1f}%)")

print("="*60)

# Cell: Free GPU Memory - Complete Cleanup

In [ ]:
# @title
import torch
import gc

print(f"Before: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
torch.cuda.empty_cache()
gc.collect()
print(f"After: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

Before: 58.87 GB
After: 0.55 GB


**Cell 7: Test on Random Images from ALL Classes**

In [ ]:
# @title
# Cell: Comprehensive test with ground truth from label files (UPDATED for current training)
import os
import random
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd
import numpy as np
from pathlib import Path
import yaml
from IPython.display import display, HTML
from ultralytics import YOLO

print("="*70)
print("🌿 COMPREHENSIVE MULTI-LEAF TESTING SYSTEM (WITH GROUND TRUTH)")
print("="*70)

# ------------------------------------------------------------------
# 1. Load the best model from the CURRENT training run
# ------------------------------------------------------------------
best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
if not os.path.exists(best_model_path):
    print(f"⚠️ best.pt not found at {best_model_path}")
    print("   Trying last.pt instead...")
    best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/last.pt'
    if not os.path.exists(best_model_path):
        raise FileNotFoundError("No model checkpoint found in training run folder.")

best_model = YOLO(best_model_path)
print(f"✅ Model loaded from: {best_model_path}")

# ------------------------------------------------------------------
# 2. Load class names from current dataset configuration
# ------------------------------------------------------------------
with open('dataset/data.yaml', 'r') as f:
    data = yaml.safe_load(f)
    classes = data['names']
print(f"✅ Model expects {len(classes)} classes")

# ------------------------------------------------------------------
# 3. Build list of test (or validation) images with ground truth labels
# ------------------------------------------------------------------
test_images_dir = Path('dataset/test/images')
test_labels_dir = Path('dataset/test/labels')
test_data = []

for img_path in list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png')):
    label_path = test_labels_dir / img_path.name.replace('.jpg', '.txt').replace('.png', '.txt')
    if label_path.exists():
        with open(label_path, 'r') as f:
            first_line = f.readline().strip()
            if first_line:
                parts = first_line.split()
                class_id = int(parts[0])
                if class_id < len(classes):
                    test_data.append({
                        'path': img_path,
                        'class_id': class_id,
                        'class_name': classes[class_id],
                        'label_file': label_path
                    })

print(f"\n📊 Dataset Statistics:")
print(f"   Total test images with ground truth: {len(test_data)}")

if len(test_data) == 0:
    print("\n⚠️ No labeled test images found. Falling back to validation images...")
    val_images_dir = Path('dataset/val/images')
    val_labels_dir = Path('dataset/val/labels')
    for img_path in list(val_images_dir.glob('*.jpg')) + list(val_images_dir.glob('*.png')):
        label_path = val_labels_dir / img_path.name.replace('.jpg', '.txt').replace('.png', '.txt')
        if label_path.exists():
            with open(label_path, 'r') as f:
                first_line = f.readline().strip()
                if first_line:
                    class_id = int(first_line.split()[0])
                    if class_id < len(classes):
                        test_data.append({
                            'path': img_path,
                            'class_id': class_id,
                            'class_name': classes[class_id],
                            'label_file': label_path
                        })
    print(f"   Using {len(test_data)} validation images instead.")

if len(test_data) == 0:
    raise RuntimeError("No labeled images found. Check your dataset splits.")

# ------------------------------------------------------------------
# 4. Select testing mode
# ------------------------------------------------------------------
print("\n" + "="*60)
print("SELECT TESTING MODE:")
print("="*60)
print("1. Quick test (8 images)")
print("2. Medium test (25 images)")
print("3. Full test (all available images)")
print("4. Custom number of images")

mode = input("\nEnter your choice (1-4, default=2): ").strip() or "2"

if mode == '1':
    num_samples = 8
elif mode == '2':
    num_samples = 25
elif mode == '3':
    num_samples = len(test_data)
elif mode == '4':
    num_samples = int(input(f"Enter number of images (max {len(test_data)}): ").strip())
    num_samples = min(num_samples, len(test_data))
else:
    num_samples = 25

num_samples = min(num_samples, len(test_data))
print(f"\n✅ Testing on {num_samples} randomly selected images...")

# ------------------------------------------------------------------
# 5. Run inference and collect results
# ------------------------------------------------------------------
samples = random.sample(test_data, num_samples)
cols = min(4, len(samples))
rows = (len(samples) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
if rows == 1 and cols == 1:
    axes = [axes]
else:
    axes = axes.flatten()

results_data = []
class_performance = defaultdict(lambda: {'correct': 0, 'total': 0, 'confidences': []})
correct = 0
no_detections = 0

for idx, sample in enumerate(samples):
    img_path = sample['path']
    actual_class = sample['class_name']
    img = cv2.imread(str(img_path))

    results = best_model(img_path, verbose=False, conf=0.25)

    best_pred = None
    best_conf = 0
    if results[0].boxes and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            pred_id = int(box.cls[0])
            conf = float(box.conf[0])
            if conf > best_conf:
                best_conf = conf
                best_pred = classes[pred_id] if pred_id < len(classes) else 'Unknown'

        is_correct = (best_pred == actual_class)
        if is_correct:
            correct += 1

        results_data.append({
            'Image': img_path.name[:50],
            'Actual': actual_class,
            'Predicted': best_pred,
            'Confidence': f"{best_conf:.1%}",
            'Correct': '✅' if is_correct else '❌'
        })

        class_performance[actual_class]['total'] += 1
        class_performance[actual_class]['correct'] += 1 if is_correct else 0
        class_performance[actual_class]['confidences'].append(best_conf)

        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        color = 'green' if is_correct else 'red'
        axes[idx].set_title(f"Actual: {actual_class[:20]}\nPred: {best_pred[:20]} ({best_conf:.0%})", fontsize=8, color=color)
        axes[idx].axis('off')
    else:
        no_detections += 1
        axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f"Actual: {actual_class[:20]}\n❌ No Detection", fontsize=8, color='orange')
        axes[idx].axis('off')

        results_data.append({
            'Image': img_path.name[:50],
            'Actual': actual_class,
            'Predicted': 'No detection',
            'Confidence': 'N/A',
            'Correct': '⚠️'
        })
        class_performance[actual_class]['total'] += 1
        class_performance[actual_class]['confidences'].append(0)

# Hide unused subplots
for idx in range(len(samples), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------------
print("\n" + "="*70)
print("📊 COMPREHENSIVE TEST RESULTS")
print("="*70)
print(f"   Total images tested: {len(samples)}")
print(f"   Correct predictions: {correct}")
print(f"   No detections: {no_detections}")
print(f"   Overall accuracy: {correct/len(samples)*100:.1f}%")
print("="*70)

df_results = pd.DataFrame(results_data)
print("\n📋 DETAILED PREDICTIONS TABLE:")
display(df_results)

print("\n" + "="*70)
print("📈 PER-CLASS PERFORMANCE BREAKDOWN")
print("="*70)
class_summary = []
for class_name, stats in sorted(class_performance.items(), key=lambda x: -x[1]['correct']/x[1]['total'] if x[1]['total'] > 0 else 0):
    acc = stats['correct']/stats['total']*100 if stats['total'] > 0 else 0
    avg_conf = sum(stats['confidences'])/len(stats['confidences']) if stats['confidences'] else 0
    bar = '█' * int(acc / 5)
    class_summary.append({
        'Disease': class_name[:40],
        'Samples': stats['total'],
        'Correct': stats['correct'],
        'Accuracy': f"{acc:.1f}% {bar}",
        'Avg Confidence': f"{avg_conf:.1%}"
    })
df_class = pd.DataFrame(class_summary)
display(df_class)

total_samples = sum(s['total'] for s in class_performance.values())
weighted_acc = sum(s['correct'] for s in class_performance.values()) / total_samples * 100 if total_samples > 0 else 0
print("\n" + "="*70)
print("📊 WEIGHTED METRICS")
print("="*70)
print(f"   Weighted Accuracy: {weighted_acc:.1f}%")
print(f"   Total Samples: {total_samples}")
print(f"   Classes with samples: {len(class_performance)}")
print("="*70)

# Save results
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
df_results.to_csv(f'test_results_{timestamp}.csv', index=False)
df_class.to_csv(f'class_performance_{timestamp}.csv', index=False)
print(f"\n💾 Results saved to test_results_{timestamp}.csv and class_performance_{timestamp}.csv")

# Optional confusion matrix (for top classes)
if len(class_performance) > 1:
    print("\n📊 Generating confusion matrix...")
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    top_classes = sorted(class_performance.keys(), key=lambda x: class_performance[x]['total'], reverse=True)[:10]
    actuals = [r['Actual'] for r in results_data if r['Actual'] in top_classes]
    predicted = [r['Predicted'] if r['Predicted'] in top_classes else 'Other' for r in results_data if r['Actual'] in top_classes]
    cm = confusion_matrix(actuals, predicted, labels=top_classes)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[c[:20] for c in top_classes],
                yticklabels=[c[:20] for c in top_classes])
    plt.title('Confusion Matrix - Top 10 Classes')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{timestamp}.png', dpi=150)
    plt.show()
    print(f"✅ Confusion matrix saved to confusion_matrix_{timestamp}.png")

print("\n" + "="*70)
print("🎉 TESTING COMPLETE!")
print("="*70)

# Below is a new cell that lets you upload one or more images, resizes them to 640×640

In [ ]:
# @title
# Cell: Upload, test, save images + display prediction table
import cv2
import matplotlib.pyplot as plt
from google.colab import files
from ultralytics import YOLO
import os
import pandas as pd
from datetime import datetime

print("="*60)
print("📤 UPLOAD AND TEST CUSTOM IMAGES (SAVE ANNOTATED IMAGES + TABLE)")
print("="*60)

# ------------------------------------------------------------------
# 1. Load model
# ------------------------------------------------------------------
try:
    best_model
except NameError:
    print("Model not found. Loading from default path...")
    best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
    if not os.path.exists(best_model_path):
        best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/last.pt'
    best_model = YOLO(best_model_path)
    print(f"✅ Model loaded from: {best_model_path}")

# ------------------------------------------------------------------
# 2. Create a session folder in Drive
# ------------------------------------------------------------------
drive_results_folder = f"/content/drive/MyDrive/test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(drive_results_folder, exist_ok=True)
print(f"📁 Results will be saved to: {drive_results_folder}")

# CSV log file
log_file = os.path.join(drive_results_folder, "predictions_log.csv")
log_columns = ["image_name", "timestamp", "class_id", "class_name", "confidence", "x1", "y1", "x2", "y2"]
if not os.path.exists(log_file):
    pd.DataFrame(columns=log_columns).to_csv(log_file, index=False)

# ------------------------------------------------------------------
# 3. Main loop
# ------------------------------------------------------------------
all_predictions = []   # to store rows for display

while True:
    print("\n📂 Please select one or more images (jpg, png, jpeg).")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded. Exiting.")
        break

    batch_predictions = []   # for this batch

    for img_name, img_bytes in uploaded.items():
        print(f"\n--- Processing: {img_name} ---")

        # Save temporary original
        temp_path = f"/tmp/{img_name}"
        with open(temp_path, "wb") as f:
            f.write(img_bytes)

        img = cv2.imread(temp_path)
        if img is None:
            print(f"❌ Could not read {img_name}")
            os.remove(temp_path)
            continue

        # Resize to 640x640
        resized = cv2.resize(img, (640, 640))
        resized_path = f"/tmp/resized_{img_name}"
        cv2.imwrite(resized_path, resized)

        # Inference
        results = best_model(resized_path, conf=0.25, verbose=False)

        # Annotated image
        annotated = results[0].plot()

        # Save annotated image to Drive
        base_name = os.path.splitext(img_name)[0]
        annotated_save_path = os.path.join(drive_results_folder, f"{base_name}_annotated.jpg")
        cv2.imwrite(annotated_save_path, annotated)
        print(f"💾 Saved annotated image: {annotated_save_path}")

        # Display annotated image
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 8))
        plt.imshow(annotated_rgb)
        plt.axis('off')
        plt.title(f"Predictions on {img_name}")
        plt.show()

        # Process detections
        if results[0].boxes and len(results[0].boxes) > 0:
            print(f"✅ Detected {len(results[0].boxes)} object(s):")
            for box in results[0].boxes:
                pred_id = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = best_model.names[pred_id]
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                print(f"   - {class_name}: {conf:.2%}")

                row = {
                    "image_name": img_name,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "class_id": pred_id,
                    "class_name": class_name,
                    "confidence": conf,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                }
                batch_predictions.append(row)
                all_predictions.append(row)
        else:
            print("⚠️ No objects detected.")
            row = {
                "image_name": img_name,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "class_id": -1,
                "class_name": "No detection",
                "confidence": 0,
                "x1": -1,
                "y1": -1,
                "x2": -1,
                "y2": -1
            }
            batch_predictions.append(row)
            all_predictions.append(row)

        # Cleanup
        os.remove(temp_path)
        os.remove(resized_path)

    # After processing the batch, display a table of predictions for this batch
    if batch_predictions:
        df_batch = pd.DataFrame(batch_predictions)
        print("\n" + "="*60)
        print(f"📊 PREDICTIONS FOR THIS BATCH ({len(batch_predictions)} detections)")
        print("="*60)
        # Show selected columns for readability
        display(df_batch[["image_name", "class_name", "confidence", "x1", "y1", "x2", "y2"]])

    # Ask to continue
    again = input("\nTest another batch of images? (y/exit): ").strip().lower()
    if again == 'exit':
        # Save all predictions to CSV
        if all_predictions:
            df_all = pd.DataFrame(all_predictions)
            df_all.to_csv(log_file, index=False)
            print(f"💾 Full prediction log saved to: {log_file}")
        print(f"✅ All annotated images saved to: {drive_results_folder}")
        print("Exiting. Goodbye!")
        break
    elif again != 'y':
        print("Invalid input. Exiting.")
        break

**Test by disease category (for comprehensive validation)**

In [ ]:
# @title
# Cell: Test by disease category (for comprehensive validation)
import os
import random
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO
import yaml

print("="*70)
print("🔬 DISEASE CATEGORY TESTING (UPDATED FOR CURRENT MODEL)")
print("="*70)

# ------------------------------------------------------------------
# 1. Load the best model from the CURRENT training run
# ------------------------------------------------------------------
best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
if not os.path.exists(best_model_path):
    best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/last.pt'
    if not os.path.exists(best_model_path):
        raise FileNotFoundError("No model checkpoint found in training run folder.")

best_model = YOLO(best_model_path)
print(f"✅ Model loaded from: {best_model_path}")

# ------------------------------------------------------------------
# 2. Load class names from dataset configuration
# ------------------------------------------------------------------
with open('dataset/data.yaml', 'r') as f:
    data = yaml.safe_load(f)
    classes = data['names']
print(f"✅ Model expects {len(classes)} classes")

# ------------------------------------------------------------------
# 3. Prepare test images (use test set, fallback to validation if needed)
# ------------------------------------------------------------------
test_dir = 'dataset/test/images'
if not os.path.exists(test_dir) or len(os.listdir(test_dir)) == 0:
    print("⚠️ No test images found, using validation images instead.")
    test_dir = 'dataset/val/images'

test_images = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
print(f"📁 Found {len(test_images)} images in {test_dir}")

# ------------------------------------------------------------------
# 4. Group images by disease category based on filename or ground truth
# ------------------------------------------------------------------
# We'll extract disease name from the image's ground truth label (most reliable)
# But for simplicity, we still fallback to filename heuristics.
test_labels_dir = test_dir.replace('images', 'labels')
image_to_gt = {}
for img in test_images:
    label_file = img.replace('.jpg', '.txt').replace('.png', '.txt')
    label_path = os.path.join(test_labels_dir, label_file)
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            first = f.readline().strip()
            if first:
                class_id = int(first.split()[0])
                if class_id < len(classes):
                    image_to_gt[img] = classes[class_id]

# Group by ground truth disease (or filename if ground truth missing)
disease_groups = defaultdict(list)
for img in test_images:
    if img in image_to_gt:
        disease_name = image_to_gt[img]
    else:
        # Fallback: guess from filename
        disease_name = "Unknown"
        for cls in classes:
            if cls in img:
                disease_name = cls.split('___')[-1] if '___' in cls else cls
                break
    disease_groups[disease_name].append(img)

print(f"\n📊 Found {len(disease_groups)} disease categories")
print("\n📋 Disease categories and sample counts:")
for disease, images in sorted(disease_groups.items())[:20]:
    print(f"   {disease[:40]:40} {len(images):4} images")

# ------------------------------------------------------------------
# 5. Let user choose a disease category
# ------------------------------------------------------------------
print("\n" + "="*60)
print("SELECT DISEASE TO TEST:")
print("="*60)

disease_list = sorted(disease_groups.keys())
for i, disease in enumerate(disease_list[:30], 1):
    print(f"   {i:2}. {disease[:45]:45} ({len(disease_groups[disease])} images)")

print(f"   {len(disease_list)+1}. Test ALL diseases (1‑2 samples per disease)")

choice = input(f"\nEnter number (1-{len(disease_list)+1}): ").strip()

if choice.isdigit():
    choice_idx = int(choice) - 1
    if choice_idx < len(disease_list):
        selected_disease = disease_list[choice_idx]
        selected_images = disease_groups[selected_disease]
        title_suffix = selected_disease
    else:
        # All diseases: take up to 2 images per disease
        selected_images = []
        for disease, imgs in disease_groups.items():
            selected_images.extend(imgs[:2])
        title_suffix = "ALL DISEASES (sample)"
else:
    selected_images = []
    title_suffix = "Unknown"

if not selected_images:
    print("❌ No images selected.")
else:
    # Limit to reasonable number for grid display
    num_to_test = min(len(selected_images), 24)
    samples = random.sample(selected_images, num_to_test)

    cols = 4
    rows = (num_to_test + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
    axes = axes.flatten()

    correct = 0
    for idx, img_file in enumerate(samples):
        img_path = os.path.join(test_dir, img_file)

        # Get actual disease from ground truth (if available)
        if img_file in image_to_gt:
            actual = image_to_gt[img_file]
        else:
            actual = "Unknown"
            for cls in classes:
                if cls in img_file:
                    actual = cls.split('___')[-1] if '___' in cls else cls
                    break

        # Predict
        results = best_model(img_path, verbose=False)

        if results[0].boxes and len(results[0].boxes) > 0:
            # Take highest confidence detection
            best_conf = 0
            best_pred = None
            for box in results[0].boxes:
                pred_id = int(box.cls[0])
                conf = float(box.conf[0])
                if conf > best_conf:
                    best_conf = conf
                    best_pred = best_model.names[pred_id]
            predicted_name = best_pred.split('___')[-1] if '___' in best_pred else best_pred

            is_correct = (predicted_name == actual)
            if is_correct:
                correct += 1

            annotated = results[0].plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(annotated_rgb)
            color = 'green' if is_correct else 'red'
            axes[idx].set_title(f"Actual: {actual[:15]}\nPred: {predicted_name[:15]} ({best_conf:.0%})",
                               fontsize=9, color=color)
            axes[idx].axis('off')
        else:
            axes[idx].imshow(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB))
            axes[idx].set_title(f"Actual: {actual[:15]}\n❌ No Detection", fontsize=9, color='orange')
            axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(num_to_test, len(axes)):
        axes[idx].axis('off')

    plt.suptitle(f"Testing: {title_suffix}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Accuracy on {len(samples)} samples: {correct}/{len(samples)} ({correct/len(samples)*100:.1f}%)")

**Cell 8: Export Model for Android (TFLite)**

In [ ]:
# @title
# Cell: Export model for Android (TF Lite, ONNX) and save to Drive + download
from google.colab import files, drive
import os
import shutil
from datetime import datetime
from ultralytics import YOLO

print("="*60)
print("📱 EXPORTING MODEL FOR ANDROID (116 CLASSES)")
print("="*60)

# ------------------------------------------------------------------
# 1. Load the best model from your current training run
# ------------------------------------------------------------------
best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/best.pt'
if not os.path.exists(best_model_path):
    best_model_path = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/weights/last.pt'
    if not os.path.exists(best_model_path):
        raise FileNotFoundError("No model checkpoint found in training run folder.")

model = YOLO(best_model_path)
print(f"✅ Loaded model from: {best_model_path}")

# ------------------------------------------------------------------
# 2. Export to TensorFlow Lite (FP16) and ONNX
# ------------------------------------------------------------------
print("\n📦 Exporting to FP16 TFLite (recommended)...")
tflite_fp16_path = model.export(format='tflite', imgsz=320, half=True, nms=True)
print(f"✅ FP16 TFLite: {tflite_fp16_path}")

print("\n📦 Exporting to INT8 TFLite (smaller, slightly lower accuracy)...")
try:
    tflite_int8_path = model.export(format='tflite', imgsz=320, int8=True, nms=True, data='dataset/data.yaml')
    print(f"✅ INT8 TFLite: {tflite_int8_path}")
except Exception as e:
    print(f"⚠️ INT8 export failed (needs calibration), skipping. Error: {e}")
    tflite_int8_path = None

print("\n📦 Exporting to ONNX (cross‑platform)...")
onnx_path = model.export(format='onnx', imgsz=320)
print(f"✅ ONNX: {onnx_path}")

# ------------------------------------------------------------------
# 3. Save exported files to Google Drive (timestamped folder)
# ------------------------------------------------------------------
drive.mount('/content/drive')
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_folder = f'/content/drive/MyDrive/PlantDisease_AndroidExport_{timestamp}'
os.makedirs(export_folder, exist_ok=True)
print(f"\n📁 Created Drive folder: {export_folder}")

# Copy all exported files
for path in [tflite_fp16_path, tflite_int8_path, onnx_path]:
    if path and os.path.exists(path):
        dest = os.path.join(export_folder, os.path.basename(path))
        shutil.copy2(path, dest)
        print(f"   Copied: {os.path.basename(path)}")

# Also copy the best.pt and class list
shutil.copy2(best_model_path, os.path.join(export_folder, 'best.pt'))
print("   Copied: best.pt")

# Save class names
with open('dataset/data.yaml', 'r') as f:
    import yaml
    classes = yaml.safe_load(f).get('names', [])
class_file = os.path.join(export_folder, 'classes.txt')
with open(class_file, 'w') as f:
    for i, cls in enumerate(classes):
        f.write(f"{i}: {cls}\n")
print(f"   Saved: classes.txt ({len(classes)} classes)")

# ------------------------------------------------------------------
# 4. Automatically download TFLite files to your computer
# ------------------------------------------------------------------
print("\n💾 Downloading TFLite models to your local machine...")
if tflite_fp16_path and os.path.exists(tflite_fp16_path):
    files.download(tflite_fp16_path)
if tflite_int8_path and os.path.exists(tflite_int8_path):
    files.download(tflite_int8_path)

# ------------------------------------------------------------------
# 5. Show file sizes
# ------------------------------------------------------------------
print("\n📊 EXPORTED FILE SIZES:")
for fname in os.listdir(export_folder):
    fpath = os.path.join(export_folder, fname)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"   {fname}: {size_mb:.1f} MB")

print("\n" + "="*60)
print("🎉 EXPORT COMPLETE!")
print("="*60)
print(f"✅ All files saved to: {export_folder}")
print("✅ TFLite files have been downloaded to your computer.")
print("="*60)

# Publication charts, barcharts and tables of our model

In [ ]:
# @title
# Cell: Generate publication charts, tables, and zip them to Drive
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime
import zipfile
import shutil
from google.colab import drive, files

# ----------------------------------------------------------------------
# 1. Mount Google Drive and create publication folder
# ----------------------------------------------------------------------
drive.mount('/content/drive')
pub_folder = '/content/drive/MyDrive/publication'
os.makedirs(pub_folder, exist_ok=True)
print(f"📁 Publication folder: {pub_folder}")

# ----------------------------------------------------------------------
# 2. Load training results (if available)
# ----------------------------------------------------------------------
results_csv = '/content/runs/detect/plant_disease_89class/yolo11x_optimized/results.csv'
training_df = None
if os.path.exists(results_csv):
    training_df = pd.read_csv(results_csv)
    print("✅ Loaded training results.csv")
else:
    print("⚠️ training results.csv not found – will generate charts without training curve")

# ----------------------------------------------------------------------
# 3. Define final metrics (from your evaluation)
# ----------------------------------------------------------------------
final_metrics = {
    'mAP50': 69.8,
    'mAP50-95': 68.4,
    'Precision': 62.2,
    'Recall': 69.5
}

# ----------------------------------------------------------------------
# 4. Comparison data (studies, mAP50, class counts)
# ----------------------------------------------------------------------
studies = [
    "Moupojou et al. (FieldPlant)",
    "PlantDoc benchmark",
    "PlantVillage (typical)",
    "Your previous run",
    "Your final model"
]
mAP50_values = [38, 45, 88, 49.3, final_metrics['mAP50']]
class_counts = [27, 27, 38, 116, 116]
colors = ['#1f77b4', '#1f77b4', '#1f77b4', '#ff7f0e', '#2ca02c']

# ----------------------------------------------------------------------
# 5. Figure 1: Bar chart comparison
# ----------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(10, 6))
bars = ax1.bar(studies, mAP50_values, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('mAP50 (%)', fontsize=12)
ax1.set_title('Comparison of Plant Disease Detection Models', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, mAP50_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add class count annotation
for i, (study, cls) in enumerate(zip(studies, class_counts)):
    ax1.text(i, 5, f'({cls} classes)', ha='center', fontsize=8, color='gray')

ax1.axhline(y=final_metrics['mAP50'], color='red', linestyle='--', linewidth=1.5,
            alpha=0.7, label=f"Your final model ({final_metrics['mAP50']:.1f}%)")
ax1.legend(loc='lower right')
plt.tight_layout()
bar_chart_path = os.path.join(pub_folder, 'comparison_bar_chart.png')
plt.savefig(bar_chart_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Bar chart saved: {bar_chart_path}")

# ----------------------------------------------------------------------
# 6. Figure 2: Training curves (mAP50 and losses) if results.csv exists
# ----------------------------------------------------------------------
if training_df is not None:
    fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(14, 5))

    # mAP50 curve
    if 'metrics/mAP50(B)' in training_df.columns:
        ax2a.plot(training_df['epoch'], training_df['metrics/mAP50(B)'], 'g-', linewidth=2)
        ax2a.set_xlabel('Epoch')
        ax2a.set_ylabel('mAP50')
        ax2a.set_title('mAP50 Progress')
        ax2a.grid(True, alpha=0.3)
        # Mark best mAP50
        best_idx = training_df['metrics/mAP50(B)'].idxmax()
        best_epoch = training_df.loc[best_idx, 'epoch']
        best_map50 = training_df.loc[best_idx, 'metrics/mAP50(B)']
        ax2a.plot(best_epoch, best_map50, 'ro', markersize=8)
        ax2a.annotate(f'Best: {best_map50:.3f}', xy=(best_epoch, best_map50),
                      xytext=(best_epoch+2, best_map50-0.05),
                      arrowprops=dict(arrowstyle='->', color='red'))

    # Loss curves
    if 'train/box_loss' in training_df.columns and 'train/cls_loss' in training_df.columns:
        ax2b.plot(training_df['epoch'], training_df['train/box_loss'], 'b-', label='Box Loss')
        ax2b.plot(training_df['epoch'], training_df['train/cls_loss'], 'r-', label='Class Loss')
        ax2b.set_xlabel('Epoch')
        ax2b.set_ylabel('Loss')
        ax2b.set_title('Training Losses')
        ax2b.legend()
        ax2b.grid(True, alpha=0.3)

    plt.tight_layout()
    train_curve_path = os.path.join(pub_folder, 'training_curves.png')
    plt.savefig(train_curve_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Training curves saved: {train_curve_path}")

# ----------------------------------------------------------------------
# 7. Figure 3: Confusion matrix (if file exists) or create a placeholder
# ----------------------------------------------------------------------
# Look for confusion matrix generated earlier
confusion_path = None
for f in os.listdir('.'):
    if f.startswith('confusion_matrix_') and f.endswith('.png'):
        confusion_path = f
        break
if confusion_path and os.path.exists(confusion_path):
    dest_conf = os.path.join(pub_folder, 'confusion_matrix.png')
    shutil.copy2(confusion_path, dest_conf)
    print(f"✅ Confusion matrix copied: {dest_conf}")
else:
    print("⚠️ No confusion matrix found – skipping")

# ----------------------------------------------------------------------
# 8. Figure 4: Dataset composition pie chart (plantwild vs FieldPlant)
# ----------------------------------------------------------------------
train_dir = 'dataset/train/images'
if os.path.exists(train_dir):
    n_plantwild = 0
    n_fieldplant = 0
    for split in ['train', 'val', 'test']:
        split_dir = f'dataset/{split}/images'
        if os.path.exists(split_dir):
            for fname in os.listdir(split_dir):
                if 'plantwild' in fname:
                    n_plantwild += 1
                elif 'fieldplant' in fname:
                    n_fieldplant += 1
    total_src = n_plantwild + n_fieldplant
    print(f"Counted from dataset: plantwild={n_plantwild}, FieldPlant={n_fieldplant}, Total={total_src}")
else:
    n_plantwild = 18533
    n_fieldplant = 5156
    total_src = n_plantwild + n_fieldplant
    print(f"Using fallback counts: plantwild={n_plantwild}, FieldPlant={n_fieldplant}")

fig4, ax4 = plt.subplots(figsize=(6,6))
sources = ['plantwild (lab / controlled)', 'FieldPlant (field images)']
counts = [n_plantwild, n_fieldplant]
colors_src = ['#ff9999', '#66b3ff']
ax4.pie(counts, labels=sources, autopct='%1.1f%%', colors=colors_src, startangle=90,
        explode=(0.05, 0), shadow=True)
ax4.set_title('Dataset Composition by Source', fontsize=14, fontweight='bold')
pie_chart_path = os.path.join(pub_folder, 'dataset_source_pie.png')
plt.savefig(pie_chart_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Dataset source pie chart saved: {pie_chart_path}")

# ----------------------------------------------------------------------
# 9. Figure 5: Train/Val/Test split bar chart
# ----------------------------------------------------------------------
if os.path.exists('dataset/train/images'):
    n_train = len(os.listdir('dataset/train/images'))
    n_val = len(os.listdir('dataset/val/images'))
    n_test = len(os.listdir('dataset/test/images'))
    total = n_train + n_val + n_test
    print(f"Split counts from folders: Train={n_train}, Val={n_val}, Test={n_test}, Total={total}")
else:
    n_train, n_val, n_test = 16582, 4737, 2370
    total = n_train + n_val + n_test
    print(f"Using fallback split counts: Train={n_train}, Val={n_val}, Test={n_test}")

fig5, ax5 = plt.subplots(figsize=(6,5))
splits = ['Training', 'Validation', 'Test']
split_counts = [n_train, n_val, n_test]
split_colors = ['#2ca02c', '#ff7f0e', '#1f77b4']
bars = ax5.bar(splits, split_counts, color=split_colors, edgecolor='black')
ax5.set_ylabel('Number of images', fontsize=12)
ax5.set_title('Dataset Split Distribution', fontsize=14, fontweight='bold')
for bar, cnt in zip(bars, split_counts):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{cnt} ({cnt/total*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax5.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
split_bar_path = os.path.join(pub_folder, 'dataset_split_bar.png')
plt.savefig(split_bar_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Dataset split bar chart saved: {split_bar_path}")

# Save split numbers as CSV
split_df = pd.DataFrame({
    'Split': splits,
    'Images': split_counts,
    'Percentage': [f"{cnt/total*100:.1f}%" for cnt in split_counts]
})
split_csv = os.path.join(pub_folder, 'dataset_split.csv')
split_df.to_csv(split_csv, index=False)
print(f"✅ Dataset split CSV saved: {split_csv}")

# ----------------------------------------------------------------------
# 10. Table 1: Overall performance (as CSV and image)
# ----------------------------------------------------------------------
overall_df = pd.DataFrame({
    'Metric': ['mAP50', 'mAP50-95', 'Precision', 'Recall'],
    'Value (%)': [final_metrics['mAP50'], final_metrics['mAP50-95'],
                  final_metrics['Precision'], final_metrics['Recall']]
})
overall_csv = os.path.join(pub_folder, 'overall_performance.csv')
overall_df.to_csv(overall_csv, index=False)
print(f"✅ Overall performance CSV: {overall_csv}")

# Create a simple table image
fig6, ax6 = plt.subplots(figsize=(5, 2))
ax6.axis('tight')
ax6.axis('off')
table_data = overall_df.values.tolist()
table = ax6.table(cellText=table_data, colLabels=overall_df.columns,
                  cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
plt.tight_layout()
table_img_path = os.path.join(pub_folder, 'overall_performance_table.png')
plt.savefig(table_img_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Overall performance table image: {table_img_path}")

# ----------------------------------------------------------------------
# 11. Table 2: Comparison table (same as the one in your paper) as CSV
# ----------------------------------------------------------------------
comparison_data = [
    ['Moupojou et al. (IEEE Access 2023)', 'FieldPlant', 27, 'Faster R-CNN', '~38%', 'Real-field images, 5,156 images'],
    ['PlantDoc benchmark', 'PlantDoc', 27, 'YOLOv5', '43-48%', 'Mixed web images, 2,569 images'],
    ['Typical PlantVillage paper', 'PlantVillage', 38, 'YOLOv5/v8', '85-90%', 'Laboratory images, single leaf'],
    ['Your previous run', 'Combined (plantwild + FieldPlant)', 116, 'YOLOv11x', '49.3% (peak)', '23,689 images, mixed lab+field'],
    ['Your final model', 'Combined (same)', 116, 'YOLOv11x', f'{final_metrics["mAP50"]:.1f}%', 'State-of-the-art for 116 real-world classes']
]
comparison_df = pd.DataFrame(comparison_data, columns=['Study', 'Dataset', 'Classes', 'Architecture', 'mAP50', 'Notes'])
comparison_csv = os.path.join(pub_folder, 'comparison_table.csv')
comparison_df.to_csv(comparison_csv, index=False)
print(f"✅ Comparison table CSV: {comparison_csv}")

# ----------------------------------------------------------------------
# 12. Zip all files in the publication folder
# ----------------------------------------------------------------------
zip_name = f'publication_materials_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
zip_path = os.path.join(pub_folder, zip_name)
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, file_list in os.walk(pub_folder):
        for file in file_list:
            if file == zip_name:  # avoid zipping the zip itself
                continue
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, pub_folder))
print(f"✅ Zip archive created: {zip_path}")

# ----------------------------------------------------------------------
# 13. Download the zip file to local machine
# ----------------------------------------------------------------------
# Ensure the files module is still accessible (no variable conflict)
from google.colab import files
files.download(zip_path)
print("📥 Zip file downloaded to your computer.")

# ----------------------------------------------------------------------
# 14. Final summary
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("🎉 PUBLICATION MATERIALS GENERATED AND SAVED!")
print("="*60)
print(f"📁 All files are in: {pub_folder}")
print(f"📦 Zip archive: {zip_path}")
print("📊 Generated files:")
for f in os.listdir(pub_folder):
    if f != zip_name:
        print(f"   - {f}")
print("="*60)